In [1]:
# ============================================================
# 1. Import
# ============================================================

import os
import zipfile
import numpy as np
import tensorflow as tf

from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


# ============================================================
# 2. Seed
# ============================================================

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)


# ============================================================
# 3. Dataset Path
# ============================================================

DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/UCI HAR Dataset"


# ============================================================
# 4. Download UCI-HAR
# ============================================================

if os.path.exists(DATA_PATH):

    print("Existing dataset found")
    print(DATA_PATH)

else:

    print("Downloading UCI-HAR dataset...")

    url = (
        "https://archive.ics.uci.edu/ml/machine-learning-databases/"
        "00240/UCI%20HAR%20Dataset.zip"
    )

    zip_path = tf.keras.utils.get_file(
        "UCI_HAR.zip",
        origin=url,
        cache_dir="/content",
        cache_subdir=""
    )

    # Google Drive에 압축 해제
    extract_path = "/content/drive/MyDrive/Colab Notebooks"

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_path)

    print("Download completed")


# ============================================================
# 5. Signal names
# ============================================================

signals = [
    "body_acc_x",
    "body_acc_y",
    "body_acc_z",
    "body_gyro_x",
    "body_gyro_y",
    "body_gyro_z",
    "total_acc_x",
    "total_acc_y",
    "total_acc_z"
]


# ============================================================
# 6. Load Dataset
# ============================================================

def load_data(split):

    X = []

    for signal in signals:

        file_path = (
            f"{DATA_PATH}/{split}/"
            f"Inertial Signals/{signal}_{split}.txt"
        )

        data = np.loadtxt(file_path)

        X.append(data)

    # (9, N, 128)
    # ->
    # (N, 128, 9)

    X = np.stack(X, axis=-1)

    y_path = f"{DATA_PATH}/{split}/y_{split}.txt"

    y = np.loadtxt(y_path).astype(int)

    # 1~6 -> 0~5
    y = y - 1

    return X, y


# ============================================================
# 7. Load Train / Test
# ============================================================

X_train, y_train = load_data("train")
X_test, y_test = load_data("test")


print("\nDataset shape")

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test :", X_test.shape)
print("y_test :", y_test.shape)


# ============================================================
# 8. Normalization
# ============================================================

mean = X_train.mean(
    axis=(0, 1),
    keepdims=True
)

std = X_train.std(
    axis=(0, 1),
    keepdims=True
)

X_train = (
    X_train - mean
) / (
    std + 1e-8
)

X_test = (
    X_test - mean
) / (
    std + 1e-8
)


# ============================================================
# 9. One-hot Encoding
# ============================================================

n_outputs = 6

y_train_onehot = to_categorical(
    y_train,
    num_classes=n_outputs
)

y_test_onehot = to_categorical(
    y_test,
    num_classes=n_outputs
)


# ============================================================
# 10. LSTM Model
# ============================================================

inputs = Input(
    shape=(128, 9)
)


# 첫 번째 LSTM
lstm1 = LSTM(
    64,
    return_sequences=True
)(inputs)


# 두 번째 LSTM
lstm2 = LSTM(
    64
)(lstm1)


# Fully Connected
dense = Dense(
    128,
    activation="relu"
)(lstm2)


# Dropout
dropout = Dropout(
    0.3
)(dense)


# Output
outputs = Dense(
    n_outputs,
    activation="softmax"
)(dropout)


# ============================================================
# 11. Model
# ============================================================

model = Model(
    inputs=inputs,
    outputs=outputs
)


model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)


model.summary()


# ============================================================
# 12. Train
# ============================================================

history = model.fit(
    X_train,
    y_train_onehot,

    epochs=20,

    batch_size=64,

    validation_split=0.2,

    shuffle=True,

    verbose=1
)


# ============================================================
# 13. Prediction
# ============================================================

y_prob = model.predict(
    X_test,
    verbose=0
)


y_pred = np.argmax(
    y_prob,
    axis=1
)


# ============================================================
# 14. Evaluation
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)


print("\n==============================")
print("LSTM TEST RESULTS")
print("==============================")

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")


# ============================================================
# 15. Classification Report
# ============================================================

class_names = [
    "WALKING",
    "WALKING_UPSTAIRS",
    "WALKING_DOWNSTAIRS",
    "SITTING",
    "STANDING",
    "LAYING"
]


print("\n==============================")
print("CLASSIFICATION REPORT")
print("==============================")


print(
    classification_report(
        y_test,
        y_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)


# ============================================================
# 16. Confusion Matrix
# ============================================================

cm = confusion_matrix(
    y_test,
    y_pred
)


print("\n==============================")
print("CONFUSION MATRIX")
print("==============================")


print(cm)


Existing dataset found
/content/drive/MyDrive/Colab Notebooks/UCI HAR Dataset

Dataset shape
X_train: (7352, 128, 9)
y_train: (7352,)
X_test : (2947, 128, 9)
y_test : (2947,)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 9)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128, 64)        │        18,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 61,062 (238.52 KB)

 Trainable params: 61,062 (238.52 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 21s 184ms/step - accuracy: 0.7114 - loss: 0.7736 - val_accuracy: 0.8083 - val_loss: 0.4715
Epoch 2/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 16s 174ms/step - accuracy: 0.9029 - loss: 0.2743 - val_accuracy: 0.9130 - val_loss: 0.2827
Epoch 3/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 17s 181ms/step - accuracy: 0.9475 - loss: 0.1506 - val_accuracy: 0.8994 - val_loss: 0.2519
Epoch 4/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 17s 180ms/step - accuracy: 0.9401 - loss: 0.1614 - val_accuracy: 0.9062 - val_loss: 0.3840
Epoch 5/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 22s 193ms/step - accuracy: 0.9524 - loss: 0.1196 - val_accuracy: 0.8973 - val_loss: 0.3884
Epoch 6/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 19s 179ms/step - accuracy: 0.9556 - loss: 0.1053 - val_accuracy: 0.8980 - val_loss: 0.3924
Epoch 7/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 17s 180ms/step - accuracy: 0.9592 - loss: 0.0961 - val_accuracy: 0.8960 - val_loss: 0.4374
Epoch 8/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 17s 182ms/step - accuracy: 0.9619 - loss: 0.0894 - val_accu